<center>
<img src="https://upload.wikimedia.org/wikipedia/fr/thumb/1/1d/Logo_T%C3%A9l%C3%A9com_SudParis.svg/1014px-Logo_T%C3%A9l%C3%A9com_SudParis.svg.png" width="10%" />
</center>

<center> <h2> NET 4103/7431 Complex Network </h2> </center>

<center> <h3> Vincent Gauthier (vincent.gauthier@telecom-sudparis.eu) </h3> </center>

### Note
Avant de commencer les exercices, assurez-vous que tout fonctionne comme prévu. Tout d'abord, le redémarrage du kernel **(dans la barre de menus, sélectionnez le kernel $\rightarrow$ Restart)**.

Assurez-vous que vous remplir les célluler aux endroits marquer «YOUR CODE HERE». 

Veuillez supprimer les ligne «raise NotImplementedError()» dans toutes les cellules auxquelles vous avez répondu, ainsi que votre nom et prénom ci-dessous:

In [ ]:
NOM = "XXX"
PRENOM = "XXX"

---

# Named entity and relation extraction with GLiNER

This tutorial uses a pretrained GLiNER model to identify named entities in a passage and infer relations between them. The results can be viewed as a small knowledge graph: entities are nodes and predicted relations are labeled edges.

## 1. Load the model and prepare text

The selected checkpoint supports relation extraction. The passage is the document to analyze; edit it to explore other examples. Model loading may download files on the first run.


In [ ]:
# Import GLiNER, a model for extracting entities and relations from text.
from gliner import GLiNER

# Load a pretrained relation-extraction model (the first run may download it).
model = GLiNER.from_pretrained("knowledgator/gliner-relex-large-v1.0")

# Provide a short passage; the model will identify spans and their relationships.
text = """
Montevideo was founded by the Spanish in the early 18th century as a military stronghold. Uruguay won its independence in 1828 following a three-way struggle between Spain, Argentina and Brazil. It is a constitutional democracy, where the president fulfills the roles of both head of state and head of government
"""

# Candidate entity types guide extraction; relation labels define edge types.
entity_labels = ["person", "organization", "location", "product", "monetary value"]
relation_labels = ["part of", "as border", "located_in"]

# Run inference on a one-document batch. Thresholds control which predictions are retained.
entities, relations = model.inference(
    [text],
    labels=entity_labels,
    relations=relation_labels,
    threshold=0.5,
    relation_threshold=0.5,
)

# Display each detected entity and its predicted type.
print("Entities:")
for entity in entities[0]:
    print(f"  {entity['text']} ({entity['label']})")

# Convert relation endpoint indices back to entity records for readable graph edges.
print("\nRelations (knowledge graph edges):")
for relation in relations[0]:
    head = entities[0][relation["head"]["entity_idx"]]
    tail = entities[0][relation["tail"]["entity_idx"]]
    print(f"  {head['text']} --[{relation['relation']}]--> {tail['text']}")


## 2. Choose labels and run inference

Entity labels and relation labels are candidate categories supplied to the model. The `threshold` and `relation_threshold` parameters filter predictions by confidence; lowering them may increase recall but also add incorrect predictions.


## 3. Read the output as a graph

The output prints detected entity spans with their labels, then resolves each relation’s entity indices into readable head–relation–tail triples. Predictions are model outputs, not verified facts: inspect the source passage and consider adjusting labels or thresholds.


In [ ]:
# Count the extracted nodes and relation edges for this document.
print(f"Entity count: {len(entities[0])}")
print(f"Relation count: {len(relations[0])}")


# Student exercises: Named Entity Recognition (NER)

Use the loaded `model` and the `text` passage from above. GLiNER uses the entity labels you provide, so label choice can affect what is recognized. Run the notebook from top to bottom before starting. For each exercise, inspect the predictions rather than assuming they are correct.

## Exercise 1 — Expand entity types

Add at least two useful entity labels to `entity_labels` (for example, `date`, `nationality`, or `government role`). Run inference again and compare the entities and labels with the original results. Which additions helped, and did any produce questionable predictions?


In [ ]:
# TODO: Add two or more labels appropriate for the passage.
student_entity_labels = entity_labels + ["date", "nationality"]

# TODO: Run entity extraction with your revised candidate labels.
student_entities, _ = model.inference(
    [text],
    labels=student_entity_labels,
    relations=relation_labels,
    threshold=0.5,
    relation_threshold=0.5,
)

# Review the detected spans and their predicted types.
for entity in student_entities[0]:
    print(f"{entity['text']} ({entity['label']})")


## Exercise 2 — Study the confidence threshold

Run the same passage at three entity thresholds. Count the predictions at each threshold. How does lowering the threshold affect the number of entities? Does it also affect the quality of the predictions?


In [ ]:
# Compare entity predictions across thresholds; keep relation threshold fixed.
for threshold in [0.3, 0.5, 0.7]:
    threshold_entities, _ = model.inference(
        [text],
        labels=entity_labels,
        relations=relation_labels,
        threshold=threshold,
        relation_threshold=0.5,
    )
    found = threshold_entities[0]
    print(f"\nThreshold {threshold:.1f}: {len(found)} entities")
    for entity in found:
        print(f"  {entity['text']} ({entity['label']})")


## Exercise 3 — Evaluate extraction against a hand-labeled reference

Create a small reference list of entity spans and labels from the passage. Compare it with model predictions using exact text-and-label matches. Calculate precision, recall, and F1. Discuss why exact matching may be too strict when predicted spans differ slightly.


In [ ]:
# TODO: Replace or extend this small reference set after reading the passage.
gold_entities = {
    ("Montevideo", "location"),
    ("Uruguay", "location"),
    ("Spain", "location"),
}
predicted_entities = {(item["text"], item["label"]) for item in entities[0]}

# Exact matches require both the entity text and its label to agree.
true_positives = len(gold_entities & predicted_entities)
false_positives = len(predicted_entities - gold_entities)
false_negatives = len(gold_entities - predicted_entities)
precision = true_positives / (true_positives + false_positives) if true_positives + false_positives else 0.0
recall = true_positives / (true_positives + false_negatives) if true_positives + false_negatives else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
print(f"Precision: {precision:.3f} | Recall: {recall:.3f} | F1: {f1:.3f}")


## Exercise 4 — Try a new document and reflect

Write a short passage about a place, person, or organization of your choice. Run GLiNER on it and report at least three extracted entities. Which predictions were useful or incorrect? How did your choice of labels affect the output?

### Optional challenge

Compare two different passages and calculate exact-match precision and recall for each using manually prepared gold entity sets. Explain why the results may vary across topics.
